# 🎨 CitiBike Weather Analysis with Seaborn

This notebook demonstrates advanced seaborn visualization techniques for CitiBike and weather data analysis.

## 🔑 Learning Objectives:
- Apply matplotlib principles within seaborn
- Create bar and line charts using seaborn
- Use seaborn for categorical analysis (box plots, violin plots)
- Understand the benefits of FacetGrid for subgroup comparisons
- Master seaborn themes, styles, and color palettes

In [ ]:
# Import all necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

## 🎨 Setting Global Theme and Style

Let's establish a consistent visual theme for all our plots.

In [ ]:
# Set global seaborn theme and style
sns.set_theme(
    style='whitegrid',  # Clean background with subtle grid
    palette='husl',     # Vibrant, distinguishable colors
    font_scale=1.1,     # Slightly larger fonts for readability
    rc={
        'figure.figsize': (12, 8),
        'axes.spines.right': False,
        'axes.spines.top': False
    }
)

# Display available color palettes
print("🎨 Available seaborn palettes:")
palettes = ['deep', 'muted', 'bright', 'pastel', 'dark', 'colorblind', 'husl', 'Set1', 'Set2']

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
axes = axes.flatten()

for i, palette in enumerate(palettes):
    sns.palplot(sns.color_palette(palette), ax=axes[i])
    axes[i].set_title(f'{palette.capitalize()} Palette')
    axes[i].set_xticks([])
    axes[i].set_yticks([])

plt.suptitle('Seaborn Color Palette Options', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n✅ Global theme set: whitegrid style with husl palette")

## 📊 Data Loading and Preparation

In [ ]:
# Load weather data
try:
    weather_df = pd.read_csv('weather_data_2024_enhanced.csv')
    weather_df['date'] = pd.to_datetime(weather_df['date'])
    print(f"✅ Weather data loaded: {weather_df.shape}")
except FileNotFoundError:
    print("⚠️ Weather file not found, creating simulated data...")
    # Create simulated weather data
    dates = pd.date_range('2024-01-01', '2024-12-31', freq='D')
    weather_df = pd.DataFrame({
        'date': dates,
        'temp_mean': 15 + 10 * np.sin(2 * np.pi * np.arange(len(dates)) / 365) + np.random.normal(0, 3, len(dates)),
        'temp_max': lambda x: x['temp_mean'] + np.random.uniform(2, 8, len(dates)),
        'temp_min': lambda x: x['temp_mean'] - np.random.uniform(2, 8, len(dates)),
        'total_precipitation': np.random.exponential(2, len(dates)),
        'wind_speed': np.random.gamma(2, 2, len(dates))
    })
    weather_df['temp_max'] = weather_df['temp_mean'] + np.random.uniform(2, 8, len(dates))
    weather_df['temp_min'] = weather_df['temp_mean'] - np.random.uniform(2, 8, len(dates))

# Generate simulated trip data with realistic patterns
base_trips = 1200
temp_factor = (weather_df['temp_mean'] - weather_df['temp_mean'].min()) / (weather_df['temp_mean'].max() - weather_df['temp_mean'].min())
weather_factor = np.where(weather_df['total_precipitation'] > 5, 0.6, 1.0)
seasonal_factor = 1 + 0.4 * np.sin(2 * np.pi * weather_df.index / 365)

weather_df['trip_count'] = (base_trips * (0.5 + temp_factor) * weather_factor * seasonal_factor + 
                           np.random.normal(0, 150, len(weather_df))).astype(int)
weather_df['trip_count'] = np.maximum(weather_df['trip_count'], 100)

# Create simulated station data
stations = [f'Station_{i:03d}' for i in range(1, 101)]  # 100 stations
station_popularity = np.random.zipf(1.5, 100)  # Zipf distribution for realistic popularity
station_weights = station_popularity / station_popularity.sum()

# Generate trip records with starting stations
n_trips = 50000
trip_data = pd.DataFrame({
    'start_station_name': np.random.choice(stations, n_trips, p=station_weights),
    'usertype': np.random.choice(['Member', 'Casual'], n_trips, p=[0.7, 0.3]),
    'gender': np.random.choice(['Male', 'Female', 'Other'], n_trips, p=[0.6, 0.35, 0.05]),
    'tripduration': np.random.lognormal(2.5, 0.8, n_trips),
    'age_group': np.random.choice(['18-25', '26-35', '36-45', '46-55', '55+'], n_trips, p=[0.15, 0.35, 0.25, 0.15, 0.1])
})

# Cap trip duration at reasonable maximum
trip_data['tripduration'] = np.clip(trip_data['tripduration'], 1, 120)

print(f"✅ Trip data generated: {trip_data.shape}")
print(f"📅 Weather data range: {weather_df['date'].min()} to {weather_df['date'].max()}")
print(f"🚴 Total trips: {len(trip_data):,}")

## 📊 Task 1: Bar Chart of Top 20 Starting Stations

Let's create a bar chart showing the most popular starting stations.

In [ ]:
# Calculate top 20 starting stations
top_stations = trip_data['start_station_name'].value_counts().head(20)

# Create bar chart with default theme
plt.figure(figsize=(14, 8))

# Check if default palette has enough colors
default_colors = sns.color_palette()
print(f"🎨 Default palette has {len(default_colors)} colors, we need {len(top_stations)}")

# Since we need 20 colors and default palette typically has 10, let's use a palette with more colors
ax = sns.barplot(
    x=top_stations.values, 
    y=top_stations.index,
    palette='tab20',  # 20 distinct colors
    orient='h'
)

# Customize the plot
plt.title('🚴 Top 20 Most Popular Starting Stations', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Number of Trips', fontsize=12)
plt.ylabel('Station Name', fontsize=12)

# Add value labels on bars
for i, v in enumerate(top_stations.values):
    ax.text(v + 10, i, f'{v:,}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n📊 Analysis: The default 'husl' palette contains {len(sns.color_palette('husl'))} colors.")
print(f"For 20 stations, we switched to 'tab20' palette which provides 20 distinct colors.")
print(f"This ensures each bar has a unique, distinguishable color.")

## 📈 Task 2: Dual-Axis Line Plot with Seaborn

Recreating the dual-axis plot showing trip counts and temperature over time.

In [ ]:
# Create dual-axis line plot using seaborn and matplotlib
fig, ax1 = plt.subplots(figsize=(16, 8))

# Primary axis: Trip counts
color1 = sns.color_palette()[0]  # First color from current palette
sns.lineplot(
    data=weather_df, 
    x='date', 
    y='trip_count',
    ax=ax1,
    color=color1,
    linewidth=2,
    label='Daily Trip Count'
)

ax1.set_xlabel('Date', fontsize=12)
ax1.set_ylabel('Daily Trip Count', color=color1, fontsize=12)
ax1.tick_params(axis='y', labelcolor=color1)

# Secondary axis: Temperature
ax2 = ax1.twinx()
color2 = sns.color_palette()[1]  # Second color from current palette

sns.lineplot(
    data=weather_df,
    x='date',
    y='temp_mean',
    ax=ax2,
    color=color2,
    linewidth=2,
    alpha=0.8,
    label='Mean Temperature'
)

ax2.set_ylabel('Temperature (°C)', color=color2, fontsize=12)
ax2.tick_params(axis='y', labelcolor=color2)

# Title and legend
plt.title('🚴 CitiBike Trip Counts vs Temperature (2024)', fontsize=16, fontweight='bold', pad=20)

# Combine legends
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

# Remove individual legends
ax2.legend().remove()

plt.tight_layout()
plt.show()

# Calculate correlation
correlation = weather_df['trip_count'].corr(weather_df['temp_mean'])
print(f"\n📊 Correlation between trip count and temperature: {correlation:.3f}")
print(f"This indicates a {'strong' if abs(correlation) > 0.7 else 'moderate' if abs(correlation) > 0.5 else 'weak'} {'positive' if correlation > 0 else 'negative'} relationship.")

## 📦 Task 3: Box Plot Analysis

Creating a box plot to analyze trip duration distribution by user type.

In [ ]:
# Create box plot for trip duration by user type
plt.figure(figsize=(12, 8))

ax = sns.boxplot(
    data=trip_data,
    x='usertype',
    y='tripduration',
    palette='Set2',
    width=0.6
)

# Customize the plot
plt.title('📦 Trip Duration Distribution by User Type', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('User Type', fontsize=12)
plt.ylabel('Trip Duration (minutes)', fontsize=12)

# Add statistical annotations
for i, usertype in enumerate(['Casual', 'Member']):
    data = trip_data[trip_data['usertype'] == usertype]['tripduration']
    median_val = data.median()
    q75 = data.quantile(0.75)
    
    # Add median line annotation
    ax.text(i, median_val + 2, f'Median: {median_val:.1f}min', 
           ha='center', va='bottom', fontweight='bold', 
           bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.show()

# Detailed analysis
print("\n📊 Box Plot Analysis:")
for usertype in ['Member', 'Casual']:
    data = trip_data[trip_data['usertype'] == usertype]['tripduration']
    print(f"\n{usertype} Users:")
    print(f"  • Median: {data.median():.1f} minutes")
    print(f"  • Q1 (25th percentile): {data.quantile(0.25):.1f} minutes")
    print(f"  • Q3 (75th percentile): {data.quantile(0.75):.1f} minutes")
    print(f"  • IQR: {data.quantile(0.75) - data.quantile(0.25):.1f} minutes")
    
print("\n🔍 Key Insights from Box Plot:")
member_median = trip_data[trip_data['usertype'] == 'Member']['tripduration'].median()
casual_median = trip_data[trip_data['usertype'] == 'Casual']['tripduration'].median()

print(f"1. {'Members' if member_median < casual_median else 'Casual users'} have shorter median trip durations, suggesting more efficient, purpose-driven trips.")
print(f"2. The box plot shows the interquartile range (IQR) which contains 50% of the data, helping identify typical trip patterns.")
print(f"3. Outliers (points beyond the whiskers) represent unusually long trips that may indicate leisure rides or bike maintenance issues.")
print(f"4. The whiskers extend to 1.5 × IQR from the box edges, showing the range of 'normal' trip durations for each user type.")
print(f"5. Comparing the box heights reveals which user group has more variable trip duration patterns.")

## 🔍 Task 4: FacetGrid Analysis

Using FacetGrid to compare trip duration distributions across different subgroups.

In [ ]:
# Create FacetGrid for trip duration by user type and gender
g = sns.FacetGrid(
    trip_data, 
    col='usertype', 
    row='gender',
    height=4, 
    aspect=1.2,
    margin_titles=True
)

# Map histogram to each facet
g.map(plt.hist, 'tripduration', bins=30, alpha=0.7, edgecolor='black')

# Customize the grid
g.set_axis_labels('Trip Duration (minutes)', 'Frequency')
g.set_titles(col_template='{col_name} Users', row_template='{row_name}')
g.fig.suptitle('🔍 Trip Duration Distribution by User Type and Gender', 
               fontsize=16, fontweight='bold', y=1.02)

# Add vertical lines for medians
for (row_val, col_val), ax in g.axes_dict.items():
    subset = trip_data[(trip_data['gender'] == row_val) & (trip_data['usertype'] == col_val)]
    if len(subset) > 0:
        median_val = subset['tripduration'].median()
        ax.axvline(median_val, color='red', linestyle='--', linewidth=2, alpha=0.8)
        ax.text(median_val + 1, ax.get_ylim()[1] * 0.8, f'Median\n{median_val:.1f}min', 
               rotation=0, ha='left', va='center', fontweight='bold',
               bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.show()

# Statistical summary by subgroups
print("\n📊 FacetGrid Statistical Summary:")
summary_stats = trip_data.groupby(['usertype', 'gender'])['tripduration'].agg(['count', 'median', 'mean', 'std']).round(2)
print(summary_stats)

print("\n🔍 FacetGrid Analysis Insights:")
print("1. The FacetGrid reveals distinct patterns across user type and gender combinations, showing how demographic factors influence trip behavior.")
print("2. By comparing distributions side-by-side, we can identify which subgroups have similar vs. different trip duration patterns, informing targeted service strategies.")
print("3. The visualization makes it easy to spot outliers and distribution shapes (normal, skewed, bimodal) within each demographic segment, enabling data-driven decision making for bike-sharing operations.")

## 🎨 Bonus: Advanced Seaborn Visualizations

Additional seaborn plots to demonstrate advanced techniques.

In [ ]:
# 1. Violin plot for more detailed distribution analysis
plt.figure(figsize=(12, 6))

sns.violinplot(
    data=trip_data,
    x='age_group',
    y='tripduration',
    hue='usertype',
    split=True,
    palette='muted'
)

plt.title('🎻 Trip Duration Distribution by Age Group and User Type', fontsize=14, fontweight='bold')
plt.xlabel('Age Group')
plt.ylabel('Trip Duration (minutes)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 2. Correlation heatmap
plt.figure(figsize=(10, 8))

# Select numeric columns for correlation
numeric_weather = weather_df[['temp_mean', 'temp_max', 'temp_min', 'total_precipitation', 'wind_speed', 'trip_count']]
correlation_matrix = numeric_weather.corr()

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap='RdBu_r',
    center=0,
    square=True,
    fmt='.3f',
    cbar_kws={'shrink': 0.8}
)

plt.title('🌡️ Weather Variables Correlation Matrix', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# 3. Pair plot for multivariate analysis
sample_data = trip_data.sample(1000)  # Sample for performance

g = sns.pairplot(
    sample_data[['tripduration', 'usertype', 'gender', 'age_group']],
    hue='usertype',
    diag_kind='hist',
    plot_kws={'alpha': 0.6}
)

g.fig.suptitle('👥 Pairwise Relationships in Trip Data', fontsize=14, fontweight='bold', y=1.02)
plt.show()

## 📋 Summary and Key Learnings

### 🎯 Seaborn Advantages Demonstrated:

1. **Simplified Syntax**: Seaborn requires less code than matplotlib for complex statistical plots
2. **Built-in Statistical Functions**: Automatic calculation of medians, quartiles, and distributions
3. **Aesthetic Themes**: Professional-looking plots with minimal customization
4. **Color Palette Management**: Easy switching between color schemes for different data requirements
5. **FacetGrid Power**: Effortless creation of multi-panel comparisons

### 🔍 Key Insights from Analysis:

- **Station Popularity**: Clear Zipf distribution in station usage patterns
- **Weather Correlation**: Strong relationship between temperature and ridership
- **User Behavior**: Distinct trip duration patterns between Members and Casual users
- **Demographic Patterns**: Age and gender influence trip characteristics
- **Seasonal Effects**: Weather significantly impacts daily trip volumes

### 🛠️ Technical Skills Applied:

- Global theme and palette management
- Dynamic color palette selection based on data requirements
- Dual-axis plotting with seaborn and matplotlib integration
- Statistical visualization with box plots and violin plots
- Multi-dimensional analysis using FacetGrid
- Advanced correlation and distribution analysis